In [ ]:
# Importar bibliotecas necessárias
import time
import pandas as pd
import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select

Parte 1 - Exploração Excel

In [106]:
# Leitura do arquivo Excel

df = pd.read_excel('Desafio_selenium_excel.xlsx')
df.head()

,ID,Original Value,Index,Date,Update value,Diff
0,1,"462,500.00",IPCA,11/11/22,NaN,NaN
1,2,"180,000.00",IGPM,4/25/23,NaN,NaN
2,3,"110,000.00",IPCA,5/4/23,NaN,NaN
3,4,"162,000.00",IGPM,7/14/23,NaN,NaN
4,5,"132,000.00",IPCA,7/14/22,NaN,NaN


In [ ]:
# Checar tipos de dados
df.dtypes

ID                  int64
Original Value        str
Index                 str
Date                  str
Update value      float64
Diff              float64
dtype: object

In [ ]:
# Ajuste formato de dados

df['Original Value'] = df['Original Value'].str.replace(',', '').astype(float)
# df['Date'] = pd.to_datetime(df['Date'])

In [109]:
df.head(5)

,ID,Original Value,Index,Date,Update value,Diff
0,1,462500.0,IPCA,11/11/22,NaN,NaN
1,2,180000.0,IGPM,4/25/23,NaN,NaN
2,3,110000.0,IPCA,5/4/23,NaN,NaN
3,4,162000.0,IGPM,7/14/23,NaN,NaN
4,5,132000.0,IPCA,7/14/22,NaN,NaN


In [110]:
df.columns

Index(['ID', 'Original Value', 'Index', 'Date', 'Update value', 'Diff'], dtype='str')

Parte 2 - Explorar o site

In [ ]:
# Iniciar o WebDriver (certifique-se de ter o driver do navegador instalado e configurado)

driver = webdriver.Chrome()

url = 'https://www3.bcb.gov.br/CALCIDADAO/publico/exibirFormCorrecaoValores.do?method=exibirFormCorrecaoValores'

driver.get(url)

time.sleep(2)

In [ ]:
# Buscar dados do site

driver.find_element(By.NAME, "valorCorrecao").text

In [ ]:
# Coleta do campos do site 

campo_indice = driver.find_element(By.ID, "selIndice")
campo_data_inicial = driver.find_element(By.NAME, "dataInicial")
campo_data_final = driver.find_element(By.NAME, "dataFinal")
campo_valor = driver.find_element(By.NAME, "valorCorrecao")

In [ ]:
# Teste de inclusão no campo de data

campo_data_inicial.clear()
#time.sleep(3)
campo_data_inicial.send_keys('052025')

In [ ]:
# Ativar o botão de cálculo

driver.find_element(By.CLASS_NAME, "botao").click()
#driver.back()

In [ ]:
# Checagem de inserição dos dados (por linha) no campo de valor do Site

for index, row in df.iterrows():
   valor = float(row['Original Value'])
   data = pd.to_datetime(row['Date'])
   data_formatada = data.strftime('%m/%Y')

   print(valor)
   print(data)
   print(data_formatada)
   print('---------')

In [ ]:
# Captura do valor final corrigido pela inflação

elementos = driver.find_elements(By.CLASS_NAME, "fundoPadraoAClaro3")
elementos[-1].text

In [ ]:
# Alteração do formato do valor para o formato na planilha

for index, row in df.iterrows():
   
   valor = float(row['Original Value'])

   valor_formatado = f"{valor:,.2f}"
   valor_formatado = valor_formatado.replace(',', 'X').replace('.', ',').replace('X', '.')

   campo_valor.clear()
   campo_valor.send_keys(str(valor_formatado))
   time.sleep(2)
   

Parte 3 - Automatizar a extração de dados web

In [114]:
for index, row in df.iterrows():
   campo_indice = driver.find_element(By.ID, "selIndice")
   campo_data_inicial = driver.find_element(By.NAME, "dataInicial")
   campo_data_final = driver.find_element(By.NAME, "dataFinal")
   campo_valor = driver.find_element(By.NAME, "valorCorrecao")

   valor = float(row['Original Value'])

   valor_formatado = f"{valor:,.2f}"
   valor_formatado = valor_formatado.replace(',', 'X').replace('.', ',').replace('X', '.')


   data = pd.to_datetime(row['Date'])
   data_formatada = data.strftime("%m%Y")

   indice = row['Index']

   if indice == 'IPCA':
    indice_site = "IPCA (IBGE) - a partir de 01/1980"
   else:
    indice_site = "IGP-M (FGV) - a partir de 06/1989"


   # Limpando campos
   #campo_indice.clear()
   campo_data_inicial.clear()
   campo_data_final.clear()
   campo_valor.clear()

   # Preenchendo campos
   campo_indice.send_keys(indice_site)
   campo_data_inicial.send_keys(data_formatada)
   campo_data_final.send_keys('012026')
   campo_valor.send_keys(str(valor_formatado))

   time.sleep(1)
   driver.find_element(By.CLASS_NAME, "botao").click()
   
   time.sleep(1)
   elementos = driver.find_elements(By.CLASS_NAME, "fundoPadraoAClaro3")
   valor_corrigido = elementos[-1].text
   valor_corrigido = valor_corrigido.replace('R$', '').replace('( REAL )', '')

   valor_corrigido = valor_corrigido.strip()

   time.sleep(1)
   driver.back()

   print(valor)
   #print(data)
   print(data_formatada)
   print(valor_corrigido)
   print(indice)
   print(indice_site)

   print('---- Concluido -----')

462500.0
112022
536.105,30
IPCA
IPCA (IBGE) - a partir de 01/1980
---- Concluido -----
180000.0
042023
184.094,15
IGPM
IGP-M (FGV) - a partir de 06/1989
---- Concluido -----
110000.0
052023
122.864,81
IPCA
IPCA (IBGE) - a partir de 01/1980
---- Concluido -----
162000.0
072023
173.769,95
IGPM
IGP-M (FGV) - a partir de 06/1989
---- Concluido -----
132000.0
072022
151.871,49
IPCA
IPCA (IBGE) - a partir de 01/1980
---- Concluido -----
135000.0
092023
146.061,87
IGPM
IGP-M (FGV) - a partir de 06/1989
---- Concluido -----


Parte 4 - Atualizar os valores na planilha

In [115]:
resultados = []

for index, row in df.iterrows():
   campo_indice = driver.find_element(By.ID, "selIndice")
   campo_data_inicial = driver.find_element(By.NAME, "dataInicial")
   campo_data_final = driver.find_element(By.NAME, "dataFinal")
   campo_valor = driver.find_element(By.NAME, "valorCorrecao")

   valor = float(row['Original Value'])

   valor_formatado = f"{valor:,.2f}"
   valor_formatado = valor_formatado.replace(',', 'X').replace('.', ',').replace('X', '.')


   data = pd.to_datetime(row['Date'])
   data_formatada = data.strftime("%m%Y")

   indice = row['Index']

   if indice == 'IPCA':
    indice_site = "IPCA (IBGE) - a partir de 01/1980"
   else:
    indice_site = "IGP-M (FGV) - a partir de 06/1989"


   # Limpando campos
   #campo_indice.clear()
   campo_data_inicial.clear()
   campo_data_final.clear()
   campo_valor.clear()

   # Preenchendo campos
   campo_indice.send_keys(indice_site)
   campo_data_inicial.send_keys(data_formatada)
   campo_data_final.send_keys('012026')
   campo_valor.send_keys(str(valor_formatado))

   time.sleep(1)
   driver.find_element(By.CLASS_NAME, "botao").click()
   
   time.sleep(1)
   elementos = driver.find_elements(By.CLASS_NAME, "fundoPadraoAClaro3")
   valor_corrigido = elementos[-1].text
   valor_corrigido = valor_corrigido.replace('R$', '').replace('( REAL )', '')

   valor_corrigido = valor_corrigido.strip()

   time.sleep(1)
   driver.back()

   resultados.append({
     'ID': row["ID"], 
     'Valor Corrigido': valor_corrigido
     })

   print(valor)
   #print(data)
   print(data_formatada)
   print(indice)
   print(indice_site)
   print(valor_corrigido)

   print('---- Concluido -----')

462500.0
112022
536.105,30
IPCA
IPCA (IBGE) - a partir de 01/1980
---- Concluido -----
180000.0
042023
184.094,15
IGPM
IGP-M (FGV) - a partir de 06/1989
---- Concluido -----
110000.0
052023
122.864,81
IPCA
IPCA (IBGE) - a partir de 01/1980
---- Concluido -----
162000.0
072023
173.769,95
IGPM
IGP-M (FGV) - a partir de 06/1989
---- Concluido -----
132000.0
072022
151.871,49
IPCA
IPCA (IBGE) - a partir de 01/1980
---- Concluido -----
135000.0
092023
146.061,87
IGPM
IGP-M (FGV) - a partir de 06/1989
---- Concluido -----


In [116]:
resultados

[{'ID': 1, 'Valor Corrigido': '536.105,30'},
 {'ID': 2, 'Valor Corrigido': '184.094,15'},
 {'ID': 3, 'Valor Corrigido': '122.864,81'},
 {'ID': 4, 'Valor Corrigido': '173.769,95'},
 {'ID': 5, 'Valor Corrigido': '151.871,49'},
 {'ID': 6, 'Valor Corrigido': '146.061,87'}]

In [117]:
df_resultados = pd.DataFrame(resultados)
df_resultados.head()

,ID,Valor Corrigido
0,1,"536.105,30"
1,2,"184.094,15"
2,3,"122.864,81"
3,4,"173.769,95"
4,5,"151.871,49"


In [121]:
df_resultados["Valor Corrigido Numero"] = (
    df_resultados["Valor Corrigido"]
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [122]:
df_resultados

,ID,Valor Corrigido,Valor Corrigido Numero
0,1,"536.105,30",536105.30
1,2,"184.094,15",184094.15
2,3,"122.864,81",122864.81
3,4,"173.769,95",173769.95
4,5,"151.871,49",151871.49
5,6,"146.061,87",146061.87


In [123]:
df_final = df.merge(
    df_resultados, 
    on='ID',
    how='left'
)

df_final.head()

,ID,Original Value,Index,Date,Update value,Diff,Valor Corrigido,Valor Corrigido Numero
0,1,462500.0,IPCA,11/11/22,NaN,NaN,"536.105,30",536105.30
1,2,180000.0,IGPM,4/25/23,NaN,NaN,"184.094,15",184094.15
2,3,110000.0,IPCA,5/4/23,NaN,NaN,"122.864,81",122864.81
3,4,162000.0,IGPM,7/14/23,NaN,NaN,"173.769,95",173769.95
4,5,132000.0,IPCA,7/14/22,NaN,NaN,"151.871,49",151871.49


In [124]:
df_final["Update value"] = df_final["Valor Corrigido Numero"]

df_final["Diff"] = (df_final["Update value"] - df_final["Original Value"])

In [125]:
df_final.head(6)

,ID,Original Value,Index,Date,Update value,Diff,Valor Corrigido,Valor Corrigido Numero
0,1,462500.0,IPCA,11/11/22,536105.30,73605.30,"536.105,30",536105.30
1,2,180000.0,IGPM,4/25/23,184094.15,4094.15,"184.094,15",184094.15
2,3,110000.0,IPCA,5/4/23,122864.81,12864.81,"122.864,81",122864.81
3,4,162000.0,IGPM,7/14/23,173769.95,11769.95,"173.769,95",173769.95
4,5,132000.0,IPCA,7/14/22,151871.49,19871.49,"151.871,49",151871.49
5,6,135000.0,IGPM,9/27/23,146061.87,11061.87,"146.061,87",146061.87


In [126]:
df_final = df_final.drop(columns=['Valor Corrigido', 'Valor Corrigido Numero'])

In [127]:
df.head(6)

,ID,Original Value,Index,Date,Update value,Diff
0,1,462500.0,IPCA,11/11/22,NaN,NaN
1,2,180000.0,IGPM,4/25/23,NaN,NaN
2,3,110000.0,IPCA,5/4/23,NaN,NaN
3,4,162000.0,IGPM,7/14/23,NaN,NaN
4,5,132000.0,IPCA,7/14/22,NaN,NaN
5,6,135000.0,IGPM,9/27/23,NaN,NaN


In [ ]:
df_final.to_excel(
    "resultado_final.xlsx",
    index=False
)

: 